# Omaha System Mapper — Interactive Evaluation & Prompt Refinement

Runs entirely inside your SageMaker notebook. No Streamlit or port access needed.

**Workflow:**
1. **Cell 1** — install deps (run once)
2. **Cell 2** — set API key + paths
3. **Cell 3** — build embedding index (cached after first run)
4. **Cell 4** — run evaluation → see F1 + error table
5. **Cell 5** — edit prompts
6. **Cell 6** — re-run with edited prompts → compare F1
7. **Cell 7** — enable verification agent for extra accuracy
8. **Cell 8** — export final `refined_prompts.py`

In [ ]:
# ── Cell 1: Install dependencies (run once) ───────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'openai', 'sentence-transformers', 'openpyxl',
                'fuzzywuzzy', 'python-Levenshtein', 'IPython'], check=False)
print('Done.')

In [ ]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────────
import os, sys

# --- EDIT THESE ---
OPENAI_API_KEY = 'sk-...'          # your OpenAI key
MODEL          = 'gpt-4o-mini'     # or 'gpt-4o' for higher accuracy
TOP_K          = 15                # retrieval top-K
USE_VERIFY     = False             # set True to enable 2nd-pass verification agent

# Paths — adjust to your EFS mount
REPO_ROOT    = '/mnt/custom-file-systems/efs/fs-0168a768a346f23a6_fsap-05d5b57d484cfde67/Zhihong.github.io/Zhihong.github.io'
OMAHA_EXCEL  = os.path.join(REPO_ROOT, 'Omaha_system list with definition.xlsx')
TRANSCRIPT   = os.path.join(REPO_ROOT, 'omaha_mapping', 'example')   # annotated TSV
# ------------------

os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
sys.path.insert(0, os.path.join(REPO_ROOT, 'omaha_mapping'))

import omaha_sagemaker as om
print('omaha_sagemaker loaded.')
print('Model configs available:', list(om.LLM_CONFIGS.keys()))

In [ ]:
# ── Cell 3: Build embedding index (cached in memory after first run) ──────────
import pandas as pd
from sentence_transformers import SentenceTransformer

if not os.path.isfile(OMAHA_EXCEL):
    raise FileNotFoundError(f'Omaha Excel not found: {OMAHA_EXCEL}')

sheets   = pd.read_excel(OMAHA_EXCEL, sheet_name=None, engine='openpyxl')
names    = list(sheets.keys())
ss_df    = sheets[names[0]].dropna(how='all').reset_index(drop=True)
int_df   = sheets[names[1]].dropna(how='all').reset_index(drop=True)
ss_df.columns  = [c.strip() for c in ss_df.columns]
int_df.columns = [c.strip() for c in int_df.columns]

print('Building embedding model…')
embed_model  = SentenceTransformer(om.EMBEDDING_MODEL, device='cpu')
ss_docs      = om.build_ss_documents(ss_df)
int_docs     = om.build_intervention_documents(int_df)
ss_emb       = om.build_index(ss_docs,  embed_model)
int_emb      = om.build_index(int_docs, embed_model)

print(f'Index ready — {len(ss_docs)} SS entries, {len(int_docs)} INT entries.')

In [ ]:
# ── Cell 4: Run evaluation ────────────────────────────────────────────────────
import warnings; warnings.filterwarnings('ignore')

def run_eval(transcript_path, model, top_k, use_verify=False,
             ss_prompt=None, int_prompt=None, label=''):
    """Run full pipeline + print F1.  Returns (metrics, df_out)."""
    # Apply prompts
    if ss_prompt  is not None: om.SS_PROMPT_TEMPLATE           = ss_prompt
    if int_prompt is not None: om.INTERVENTION_PROMPT_TEMPLATE = int_prompt

    df = pd.read_csv(transcript_path, sep='\t', dtype=str).fillna('')
    df.columns = [c.strip() for c in df.columns]

    orig_k = om.TOP_K_RETRIEVAL
    om.TOP_K_RETRIEVAL = top_k

    print(f'Running {len(df)} rows with {model}  [top_k={top_k}]  {label}…')
    df_out = om.process_sheet(df, ss_docs, ss_emb, int_docs, int_emb,
                              embed_model, model)

    if use_verify:
        print('  → running verification agent…')
        df_out = om.verify_sheet(df_out, ss_docs, ss_emb, int_docs, int_emb,
                                 embed_model, model, top_k)

    om.TOP_K_RETRIEVAL = orig_k
    metrics = om.evaluate_sheet(df_out)
    return metrics, df_out


def _prf_line(prefix, p, r, f1, tp=None, fp=None, fn=None, tn=None):
    counts = ''
    if tp is not None: counts += f'  TP={tp} FP={fp} FN={fn}'
    if tn is not None: counts += f'  TN={tn}'
    return f'  {prefix:<16}│ P={p:.3f}  R={r:.3f}  F1={f1:.3f}{counts}'


def _sub_block(title, m, pos_p, pos_r, pos_f1, none_p, none_r, none_f1,
               macro_p, macro_r, macro_f1, tp, fp, fn, tn):
    lines = [f'  {title}']
    lines.append(f'    Positive      │ P={pos_p:.3f}  R={pos_r:.3f}  F1={pos_f1:.3f}'
                 f'  TP={tp} FP={fp} FN={fn}')
    lines.append(f'    None class    │ P={none_p:.3f}  R={none_r:.3f}  F1={none_f1:.3f}'
                 f'  TN={tn}')
    lines.append(f'    ★ Macro       │ P={macro_p:.3f}  R={macro_r:.3f}  F1={macro_f1:.3f}')
    return '\n'.join(lines)


def show_metrics(metrics, label=''):
    w = 68
    print(f'\n{"═"*w}')
    if label: print(f'  {label}')

    # ── SIGNS / SYMPTOMS ─────────────────────────────────────────────────────
    print(f'\n  SIGNS / SYMPTOMS')
    print(_prf_line('Positive class',
                    metrics['ss_precision'], metrics['ss_recall'], metrics['ss_f1'],
                    tp=metrics['ss_tp'], fp=metrics['ss_fp'], fn=metrics['ss_fn']))
    print(_prf_line('None class',
                    metrics['ss_none_p'], metrics['ss_none_r'], metrics['ss_none_f1'],
                    tn=metrics['ss_tn']))
    print(_prf_line('★ Macro',
                    metrics['ss_macro_p'], metrics['ss_macro_r'], metrics['ss_macro_f1']))
    print(f'  {"Accuracy":<16}│ {metrics["ss_accuracy"]:.3f}'
          f'  ({metrics["ss_tp"]+metrics["ss_tn"]} / {metrics["ss_tp"]+metrics["ss_fp"]+metrics["ss_fn"]+metrics["ss_tn"]} turns)')

    print()
    print(_sub_block('Problem only',  metrics,
                     metrics['prob_precision'],    metrics['prob_recall'],    metrics['prob_f1'],
                     metrics['prob_none_p'],        metrics['prob_none_r'],    metrics['prob_none_f1'],
                     metrics['prob_macro_p'],       metrics['prob_macro_r'],   metrics['prob_macro_f1'],
                     metrics['prob_tp'], metrics['prob_fp'], metrics['prob_fn'], metrics['prob_tn']))

    print()
    print(_sub_block('Sign/Symptom only', metrics,
                     metrics['ss_only_precision'], metrics['ss_only_recall'], metrics['ss_only_f1'],
                     metrics['ss_only_none_p'],     metrics['ss_only_none_r'], metrics['ss_only_none_f1'],
                     metrics['ss_only_macro_p'],    metrics['ss_only_macro_r'],metrics['ss_only_macro_f1'],
                     metrics['ss_only_tp'], metrics['ss_only_fp'], metrics['ss_only_fn'], metrics['ss_only_tn']))

    # ── INTERVENTIONS ────────────────────────────────────────────────────────
    print(f'\n  INTERVENTIONS')
    print(_prf_line('Positive class',
                    metrics['int_precision'], metrics['int_recall'], metrics['int_f1'],
                    tp=metrics['int_tp'], fp=metrics['int_fp'], fn=metrics['int_fn']))
    print(_prf_line('None class',
                    metrics['int_none_p'], metrics['int_none_r'], metrics['int_none_f1'],
                    tn=metrics['int_tn']))
    print(_prf_line('★ Macro',
                    metrics['int_macro_p'], metrics['int_macro_r'], metrics['int_macro_f1']))
    print(f'  {"Accuracy":<16}│ {metrics["int_accuracy"]:.3f}'
          f'  ({metrics["int_tp"]+metrics["int_tn"]} / {metrics["int_tp"]+metrics["int_fp"]+metrics["int_fn"]+metrics["int_tn"]} turns)')

    print()
    print(_sub_block('Category only', metrics,
                     metrics['cat_precision'], metrics['cat_recall'], metrics['cat_f1'],
                     metrics['cat_none_p'],     metrics['cat_none_r'], metrics['cat_none_f1'],
                     metrics['cat_macro_p'],    metrics['cat_macro_r'],metrics['cat_macro_f1'],
                     metrics['cat_tp'], metrics['cat_fp'], metrics['cat_fn'], metrics['cat_tn']))

    print()
    print(_sub_block('Target only', metrics,
                     metrics['tgt_precision'], metrics['tgt_recall'], metrics['tgt_f1'],
                     metrics['tgt_none_p'],     metrics['tgt_none_r'], metrics['tgt_none_f1'],
                     metrics['tgt_macro_p'],    metrics['tgt_macro_r'],metrics['tgt_macro_f1'],
                     metrics['tgt_tp'], metrics['tgt_fp'], metrics['tgt_fn'], metrics['tgt_tn']))

    print(f'\n{"═"*w}')

    # Gap to target
    target = 0.9
    gaps = []
    if metrics['ss_f1']  < target: gaps.append(f'SS gap={target - metrics["ss_f1"]:.3f}')
    if metrics['int_f1'] < target: gaps.append(f'INT gap={target - metrics["int_f1"]:.3f}')
    if gaps:
        print(f'  ⚠ Remaining: {"  ".join(gaps)}')
    else:
        print(f'  ✓ Both SS and INT F1 ≥ 0.90 — target reached!')

    return metrics['ss_f1'], metrics['int_f1']


def show_errors(df_out, max_rows=40):
    """Print FP/FN rows as a pandas table."""
    rows = []
    for _, row in df_out.iterrows():
        h_ss  = [str(row.get(f'OS_SS_{j}','')).strip() for j in range(1,4)
                 if str(row.get(f'OS_SS_{j}','')).strip() not in ('','nan')]
        p_ss  = [str(row.get(f'Pred_SS_{j}','')).strip() for j in range(1,4)
                 if str(row.get(f'Pred_SS_{j}','')).strip() not in ('','nan')]
        h_int = [str(row.get(f'OS_I_{j}','')).strip() for j in range(1,4)
                 if str(row.get(f'OS_I_{j}','')).strip() not in ('','nan')]
        p_int = [str(row.get(f'Pred_I_{j}','')).strip() for j in range(1,4)
                 if str(row.get(f'Pred_I_{j}','')).strip() not in ('','nan')]
        ss_tp, ss_fp, ss_fn  = om.row_level_match(p_ss,  h_ss)
        it_tp, it_fp, it_fn  = om.row_level_match(p_int, h_int)
        if ss_fp or ss_fn or it_fp or it_fn:
            rows.append({
                'Turn': str(row.get('Conversation',''))[:110],
                'SS expected':  ' | '.join(h_ss)  or '—',
                'SS predicted': ' | '.join(p_ss)  or '—',
                'SS err': ('FN ' if ss_fn else '')+('FP' if ss_fp else ''),
                'INT expected':  ' | '.join(h_int) or '—',
                'INT predicted': ' | '.join(p_int) or '—',
                'INT err': ('FN ' if it_fn else '')+('FP' if it_fp else ''),
            })
    if rows:
        from IPython.display import display
        err_df = pd.DataFrame(rows[:max_rows])
        pd.set_option('display.max_colwidth', 120)
        display(err_df)
        print(f'Total error rows: {len(rows)}')
    else:
        print('No errors — perfect match!')


# --- RUN ---
metrics, df_out = run_eval(TRANSCRIPT, MODEL, TOP_K,
                            use_verify=USE_VERIFY, label='baseline prompts')
ss_f1, int_f1 = show_metrics(metrics, 'Baseline')
show_errors(df_out)

In [ ]:
# ── Cell 5: Load refined prompts and re-run ───────────────────────────────────
# Option A — load from refined_prompts.py (edit that file, then re-run this cell)
from refined_prompts import MY_SS_PROMPT, MY_INT_PROMPT

metrics_r, df_out_r = run_eval(TRANSCRIPT, MODEL, TOP_K,
                                use_verify=USE_VERIFY,
                                ss_prompt=MY_SS_PROMPT,
                                int_prompt=MY_INT_PROMPT,
                                label='refined prompts')
ss_f1_r, int_f1_r = show_metrics(metrics_r, 'Refined')
show_errors(df_out_r)

In [ ]:
# ── Cell 6: Edit prompts directly here and re-run ─────────────────────────────
# Copy your current best prompt from Cell 5, tweak it, then run this cell.
# TIP: look at the error rows above and add/fix examples to address FP/FN.

MY_SS_PROMPT_EDITED = MY_SS_PROMPT   # <- paste your edited SS prompt here

MY_INT_PROMPT_EDITED = MY_INT_PROMPT  # <- paste your edited INT prompt here

metrics_e, df_out_e = run_eval(TRANSCRIPT, MODEL, TOP_K,
                                use_verify=USE_VERIFY,
                                ss_prompt=MY_SS_PROMPT_EDITED,
                                int_prompt=MY_INT_PROMPT_EDITED,
                                label='edited prompts')
ss_f1_e, int_f1_e = show_metrics(metrics_e, 'Edited')
show_errors(df_out_e)

In [ ]:
# ── Cell 7: Compare all runs side-by-side ────────────────────────────────────
runs = []
try: runs.append({'Run': 'Baseline',  'SS F1': ss_f1,   'INT F1': int_f1})
except: pass
try: runs.append({'Run': 'Refined',   'SS F1': ss_f1_r, 'INT F1': int_f1_r})
except: pass
try: runs.append({'Run': 'Edited',    'SS F1': ss_f1_e, 'INT F1': int_f1_e})
except: pass

from IPython.display import display
cmp_df = pd.DataFrame(runs).set_index('Run')
cmp_df['SS ≥0.9'] = cmp_df['SS F1'].apply(lambda x: '✓' if x >= 0.9 else f'gap {0.9-x:.3f}')
cmp_df['INT ≥0.9'] = cmp_df['INT F1'].apply(lambda x: '✓' if x >= 0.9 else f'gap {0.9-x:.3f}')
display(cmp_df)

In [ ]:
# ── Cell 8: Test a single turn ────────────────────────────────────────────────
import re

def test_turn(turn, context='', model=MODEL, top_k=TOP_K, use_verify=USE_VERIFY):
    orig_k = om.TOP_K_RETRIEVAL
    om.TOP_K_RETRIEVAL = top_k

    # SS
    ss_ret   = om.retrieve(turn, ss_docs, ss_emb, embed_model)
    ss_p     = om.build_ss_prompt(turn, context, ss_ret)
    if om.LLM_CONFIGS[model]['provider'] != 'local':
        ss_p = re.sub(r'<\|[^|>]+\|>', '', ss_p).strip()
    ss_raw   = om.call_llm(ss_p, model)
    ss_preds = om.parse_ss_output(ss_raw)

    # INT
    int_ret   = om.retrieve(turn, int_docs, int_emb, embed_model)
    int_p     = om.build_intervention_prompt(turn, context, int_ret)
    int_raw   = om.call_llm(int_p, model)
    int_preds = om.parse_intervention_output(int_raw)

    # Optional verify
    if use_verify and ss_preds:
        ss_preds = om._verify_ss(turn, ss_preds, ss_docs, ss_emb, embed_model, model)
    if use_verify and int_preds:
        int_preds = om._verify_int(turn, context, int_preds, int_docs, int_emb, embed_model, model)

    om.TOP_K_RETRIEVAL = orig_k

    print('TURN:', repr(turn))
    print()
    print('── SS ──')
    if ss_preds:
        for p in ss_preds:
            print(f'  {p["domain"]} | {p["problem"]} | {p["ss"]}')
    else:
        print('  None')
    print()
    print('── INT ──')
    if int_preds:
        for p in int_preds:
            print(f'  {p["category"]} | {p["target"]}')
    else:
        print('  None')
    print()
    print('Raw SS  :', ss_raw)
    print('Raw INT :', int_raw)


# --- EXAMPLE: change the turn below and re-run ---
test_turn(
    turn='My blood pressure machine says 158 over 92.',
    context='Nurse: Can you check your blood pressure for me?',
)

In [ ]:
# ── Cell 9: Export final prompts to refined_prompts.py ────────────────────────
# Change BEST_SS / BEST_INT to whichever version had the highest F1.

BEST_SS  = MY_SS_PROMPT_EDITED   # or MY_SS_PROMPT, or a manual string
BEST_INT = MY_INT_PROMPT_EDITED  # or MY_INT_PROMPT, or a manual string

out_path = os.path.join(REPO_ROOT, 'omaha_mapping', 'refined_prompts.py')

ss_safe  = BEST_SS.replace('"""', "'''")
int_safe = BEST_INT.replace('"""', "'''")

content = f'''\"""
Refined Omaha System prompt templates.
Exported from omaha_eval.ipynb after iterative refinement.

Usage:
    python eval_local.py --prompts refined [--verbose]
"""

MY_SS_PROMPT = """{ss_safe}"""

MY_INT_PROMPT = """{int_safe}"""
'''

with open(out_path, 'w') as f:
    f.write(content)

print(f'Saved to {out_path}')
print('\nNow run:  python eval_local.py --prompts refined --verbose')